# Lesson 3.2: Visualizing Data with Plotly

**🎬 Video:** [Lesson 3.2: Visualizing Data with Plotly](#)

## Overview

In this lesson you will use the cleaned DataFrame from Lesson 3.1 to build interactive visualizations with Plotly. You will group time-series data, create line and bar charts, and perform keyword analysis.

**Prerequisites:** Complete [Lesson 3.1 — Loading and Cleaning Data](lesson_3_1_loading_and_cleaning.ipynb) first. This lesson loads the pickle file saved at the end of that lesson.

**What you will cover:**
- Load a saved pickle file
- Group datetime data into periods with `.dt.to_period()`
- Create a line chart with `px.line()`
- Analyze text with keyword matching
- Create a bar chart with `px.bar()`

---

## 📖 1 Follow Along — Loading Libraries and Data

You do not need to write or modify any code in this section. Run each cell and focus on understanding what the code is doing and why.

As in the previous lesson, we need to load our libraries before we can do anything. We also load the cleaned pickle file we saved at the end of Lesson 3.1 — this means we can skip all the data cleaning steps and go straight to visualization.

- Click the <img src="../lesson_assets/images/jupyter/run.svg" style="height: 14pt; vertical-align: middle;"> button below to load libraries and data

In [ ]:
# Import required libraries
import pandas as pd
import plotly.express as px

# Load the cleaned data from Lesson 3.1
df = pd.read_pickle("../data/JMU/JMU_raw_cleaned.pickle")
df.head()

> 👉 **Note:** *We are expanding our common development pattern for scripts. First, load libraries. Second, load data.*

---

## 2 Data Visualization

Now that we have clean data with proper data types, we can create visualizations that reveal patterns in the r/JMU subreddit.

### 2.1 Why Type Conversion Makes This Possible

This is where the type conversion we did earlier pays off. By converting the `date` column from a plain text string into an actual datetime object, pandas now understands what the values mean:

```
2024-03-18 12:50:05
```

- `2024` → year  
- `03` → March  
- `18` → day  

To prepare the data for a chart we need three steps:

1. **Group dates into periods** — `.dt.to_period('M')` collapses every date in March 2024 into the single label `2024-03`
2. **Convert back to timestamps** — `.dt.to_timestamp()` converts those labels into a format Plotly can display on a time axis
3. **Count rows per group** — `.groupby().size()` counts how many posts fall into each interval

```python
df['interval'] = df['date'].dt.to_period('M')
df['interval'] = df['interval'].dt.to_timestamp()
interval_counts = df.groupby(['interval', 'type'], as_index=False, observed=True).size()
```

> 👉 **Note:** *Steps 1 and 2 can be written as a single chained line: `df['date'].dt.to_period('M').dt.to_timestamp()`. Chaining is common in pandas code you'll find online, but breaking it into separate steps makes each operation easier to follow.*

### 2.2 Why Code Beats a Spreadsheet Here

In Excel or Google Sheets, producing a posts-per-month summary would require several manual steps: adding a helper column with a `TEXT()` or `MONTH()`/`YEAR()` formula, building a pivot table, and updating everything every time the data changes.

In Python, those three lines above do all of that. Want weekly counts instead? Change `'M'` to `'W'`. Yearly? Use `'Y'`. The column is named `interval` rather than `year_month` so the name stays accurate no matter what period you choose. The rest of the code stays exactly the same.

In [ ]:
# Step 1: Convert the date column into monthly periods (e.g. 2024-03)
df["interval"] = df["date"].dt.to_period("M")

# Step 2: Convert the periods back to timestamps so Plotly can read them
df["interval"] = df["interval"].dt.to_timestamp()

# Step 3: Group by interval and post type, then count rows in each group
# as_index=False returns a plain DataFrame — no extra steps needed
interval_counts = df.groupby(["interval", "type"], as_index=False, observed=True).size()

interval_counts.head()

> 📊 **Output:** You should see a table with three columns: `interval` (the first day of each month), `type` (either `post` or `comment`), and `size` (how many of that type occurred that month). Each month appears for both posts and comments.

### 2.3 Visualizing Posts by Month

Plotly chart functions like `px.line()` accept many keyword arguments. Each keyword controls a different aspect of the chart. You only need to include the ones you want; everything else uses a sensible default. The more arguments you pass, the more customized the result. The most common mistake is usually a syntax error. This is when you type a value or command in a way the program is not expecting.

**Syntax rules to keep in mind:**

- **Each argument is followed by a comma** — except the very last one before the closing `)`. A missing or extra comma is one of the most common errors.
- **String values go in quote marks** — column names like `"interval"` and titles like `"My Chart"` must be wrapped in quotes. Numbers and variables do not need quotes.
- **`labels=` takes a dictionary `{}`** — a dictionary is a set of `key: value` pairs indicated. Here it maps column names (the keys) to the display labels you want on the chart (the values):

```python
labels= {"size": "Number of Posts/Comments", "interval": "Time Period"}
#            ^key          ^value                  ^key          ^value
```

Each pair is separated by a comma, and the key and value are separated by a colon. Think of it as a lookup table: "whenever pandas sees the column name `size`, show the label `Number of Posts/Comments` instead." See [lesson 2.1](../lesson_2_very_basic_python/lesson_2_1_overview_variables.ipynb) for review.

In [ ]:
# Create an interactive line chart
fig = px.line(
    interval_counts,
    x="interval",  # x-axis: time
    y="size",  # y-axis: number of posts/comments
    color="type",  # separate lines for posts vs comments
    title="Number of Posts and Comments per Month on r/JMU",  # set text for title
    labels={
        "interval": "Time Period",
        "size": "Number of Posts/Comments",
    },  # control text on hover label
    markers=True,  # add dots to the lines
)

# Customize the chart appearance
fig.update_layout(
    xaxis_title="Time Period", # set x-axis title
    yaxis_title="Number of Posts/Comments", # set y-axis title
    legend_title="Type" # set legend title
)

# Display the interactive chart
fig.show()

> 💡 **Reflection:** Look at the chart around August 2020. You should see a noticeable spike in activity. What do you think caused it? Consider what was happening in the world and on college campuses at that time.

### 2.1 Keyword Analysis: How Do We Measure Popularity?

In the chart above we looked at the posting and commenting trends over time. This told us when people were active, but not *what* they were talking about. Since we have all the text people wrote, we can figure out what is driving the conversation. That is, what topics are *popular*? 

But first, a question worth thinking about: **what does "popular" actually mean?** One way Reddit data gives us insight into campus conversations is that it not only shows *what* people are talking about, but also how others are responding to it through *upvotes* and *downvotes*. 

Using this data, there are several ways you could measure whether a topic is popular on Reddit:

- **Post count** — how many posts mention the keyword at all
- **Total score** — the sum of all upvotes across every post that mentions it
- **Average score** — the typical upvote count for a post that mentions it

These can tell very different stories. Only counting the number of times a keyword appears is not a great metric, because you are not measuring how other readings are *engaging* with that topic. Meanwhile, a total *upvote* score can also be misleading. A common topic like `class` might appear in thousands of posts, racking up a high post count score, but each individual post might barely gets noticed. Meanwhile `tuition` might come up rarely, yet every time it does, the community reacts strongly, which gives it a high *average score*. 

> 👉 **Note:** *In all cases, we are only looking at literal words as they occur and not what they might mean. Phrases like, "She has no class" and "He did not have class today" both use the word class but in the former it is about someone's character and in the latter it is about actually attending class. Likewise, someone who writes about "classes being too expensive" is actually writing about `tuition`. We have to remember that these measures are all very crude at this point.*

The code below calculates both the total score and average score and plots them as separate charts. Run it and look at both. How does the ranking change depending on the method?

In [ ]:
# Define keywords to analyze - common JMU-related topics
keywords = ["tuition", "covid", "party", "football", "class", "library", "campus"]

# Create a helper function to check if text contains a keyword (case-insensitive)
def contains_keyword(text, keyword):
    # Handle missing/empty text
    if pd.isna(text):
        return False
    # Convert both to lowercase to make the search case-insensitive
    return keyword.lower() in text.lower()

# Calculate both total and average score for each keyword
keyword_scores = []

for keyword in keywords:
    # Boolean mask: True for every post that contains this keyword
    has_keyword_mask = df["text"].apply(lambda x: contains_keyword(x, keyword))
    matching_posts = df[has_keyword_mask]

    keyword_scores.append({
        "keyword": keyword,
        "total_score": matching_posts["score"].sum(),
        "avg_score": matching_posts["score"].mean()
    })

# Convert results to a DataFrame for plotting
keyword_df = pd.DataFrame(keyword_scores)

# Chart 1: Total score — reflects both how often a topic is discussed AND how well each post scores
fig1 = px.bar(keyword_df,
              x="keyword",
              y="total_score",
              title="Total Score by Keyword",
              labels={
                  "total_score": "Total Score",
                  "keyword": "Keyword",
              })
fig1.show()

# Chart 2: Average score — reflects how well a typical post about this topic is received
fig2 = px.bar(keyword_df,
              x="keyword",
              y="avg_score",
              title="Average Score by Keyword",
              labels={
                  "avg_score": "Average Score",
                  "keyword": "Keyword",
              })
fig2.show()

> 💡 **Reflection:** Do the two charts tell the same story? Find one keyword that ranks differently between them and explain why the two metrics give different results. Think about which chart you would choose if you wanted to argue that a topic is "popular" — and whether that choice is defensible.


### 2.2 Keywords Over Time

The bar charts above show keyword popularity as a single aggregate number. This is good for a quick overview, but it can also be distorting because it collapses the entire history of that word on the thread into one bar. A topic like `covid` probably does not appear before 2020, then spikes sharply and fades. A topic like `tuition` might resurface every year when bills go out.

To reveal that story, we track each keyword's mention rate across time using the same `.dt.to_period()` grouping from the line chart earlier.

**Why normalize per 1,000 posts?**

Post volume on a subreddit is not constant — there are a few dozen posts a month in early years and hundreds in peak years. If we just counted raw hits, a keyword that shows up twice in a thin early month would look identical to one that shows up twice in a busy recent month. Those are very different situations.

Dividing by total posts per interval and multiplying by 1,000 puts every period on a level playing field: *"of every 1,000 posts in this quarter, how many mentioned this keyword?"*

**Avoiding false spikes**

When a period has very few posts total, even one or two keyword mentions can produce a huge normalized rate. For example: 1 mention ÷ 2 total posts × 1,000 = 500 per 1,000 — which looks like a massive spike but is statistically meaningless. The `MIN_PERIOD_POSTS` threshold drops any period below that cutoff before the rate is calculated.


In [ ]:

# ── Keywords to track — change this list ─────────────────────────────────
KEYWORDS_OVER_TIME = ["tuition", "covid", "football"]

# ── Time interval ─────────────────────────────────────────────────────────
# 'W' = weekly, 'M' = monthly, 'Q' = quarterly, 'Y' = yearly
# Coarser intervals (Q, Y) smooth the chart by pooling more posts per bucket.
INTERVAL = "M"

# ── Minimum posts per interval ────────────────────────────────────────────
# Intervals below this threshold are dropped before normalizing
# to avoid false spikes caused by very small post counts.
MIN_PERIOD_POSTS = 30

# ── Group all posts into intervals ────────────────────────────────────────
_df_trend = df.copy()
_df_trend["interval"] = _df_trend["date"].dt.to_period(INTERVAL).dt.to_timestamp()

# Count total posts per interval — this is the denominator for normalization
_period_totals = (
    _df_trend.groupby("interval")
             .size()
             .reset_index(name="_total")
)
_period_totals = _period_totals[_period_totals["_total"] >= MIN_PERIOD_POSTS]

# ── Count keyword mentions per interval, then normalize per 1,000 ─────────
_rows = []
for kw in KEYWORDS_OVER_TIME:
    _mask = _df_trend["text"].str.contains(kw, case=False, na=False)
    _hits = (
        _df_trend[_mask]
        .groupby("interval")
        .size()
        .reset_index(name="_hits")
    )
    _m = _hits.merge(_period_totals, on="interval")
    _m["per_1000"] = _m["_hits"] / _m["_total"] * 1000
    _m["keyword"] = kw
    _rows.append(_m[["interval", "keyword", "per_1000"]])

_keyword_trend_df = pd.concat(_rows, ignore_index=True)

# ── Plot ──────────────────────────────────────────────────────────────────
fig = px.line(
    _keyword_trend_df,
    x="interval",
    y="per_1000",
    color="keyword",
    title="Keyword Mentions Over Time on r/JMU",
    labels={
        "interval": "Time Period",
        "per_1000": "Mentions per 1,000 Posts/Comments",
        "keyword": "Keyword",
    },
    markers=True,
)
fig.update_layout(
    xaxis_title="Time Period",
    yaxis_title="Mentions per 1,000 Posts/Comments",
)
fig.show()


### ✍️ 2.3 Critical Activity - Exploring Keyword Trends Over Time

Try swapping in two or three keywords of your own — pick topics you noticed while skimming the data in Lesson 3.1. Then experiment with `INTERVAL`: change it to `'M'` (monthly) and observe how the chart becomes noisier. Change it to `'Y'` (yearly) and observe how the same trends become smoother but lose fine detail. Which interval do you find most readable for this data? Likewise, changing the `MIN_PERIOD_POST` variable can increase or decrease how much data is included and make the data more and less noisy.


---

## 3 Advantages of Using Code vs. Spreadsheet Software

While the initial learning curve for using Python for data visualization is much steeper than with spreadsheet software, in the long run it is a time saver. It is much quicker to change one or two variables in a Plotly function than to reconfigure the data in Excel and create a whole new chart.

Here are a few more reasons why code has the edge:

- **Reproducibility** — Every step of your analysis is written down in order. Anyone (including future-you) can re-run the notebook and get the exact same result. A spreadsheet full of manual edits and hidden formulas is much harder to audit or repeat.

- **Scalability** — This notebook works on 10 rows or 10 million rows without any changes to the code. Spreadsheets slow down and become error-prone as data grows.

- **Easy updates** — When new Reddit data comes in next month, you re-run the notebook. In a spreadsheet, you would need to manually paste the new data, check formulas, and rebuild charts.

- **Reusable logic** — Functions and methods like `.dt.to_period('M')` are written once and called everywhere. In a spreadsheet, the same formula has to be copied into hundreds of cells.

- **Version control** — Code files can be tracked with Git, so you can see exactly what changed and when. Spreadsheet version histories are limited and hard to compare. If someone in your group changed a formula, you may never know why you are seeing a different result.

---

## Lesson Summary

Here is what you covered in this lesson:

### Part 1: Grouping Time Data
- **`.dt.to_period("M")`** — group datetime values into month periods
- **`.groupby()`** / **`.size()`** — count rows per group
- **`.dt.to_timestamp()`** — convert periods back to datetimes for Plotly

### Part 2: Line Charts
- **`px.line()`** — create an interactive line chart
- **`color=`** — split data into separate series by category
- **`fig.update_layout()`** — customize axis titles, legend, and styling

### Part 3: Keyword Analysis
- **Custom functions with `.apply()`** — run a check row-by-row
- **Boolean masking** — filter rows where text contains a keyword
- **`.groupby().mean()`** — calculate average values per group
- **`px.bar()`** — create an interactive bar chart

### Part 4: Keywords Over Time
- **Normalizing per 1,000 posts** — divide keyword hits by total posts per interval, then multiply by 1,000, so periods with different post volumes are directly comparable
- **`MIN_PERIOD_POSTS` threshold** — drop low-volume intervals before normalizing to avoid misleading spikes
- **`INTERVAL` parameter** — coarser intervals (`"Q"`, `"Y"`) produce smoother lines; finer intervals (`"W"`, `"M"`) show more detail but more noise
- **`pd.concat()`** — combine per-keyword DataFrames into one for plotting

---

In the next lesson you will practice customizing Plotly charts and build new visualizations from scratch.

➡️ **Next:** [Lesson 3.3 — Plotly Practice](lesson_3_3_plotly_practice.ipynb)
